In [1]:
#importing libararies
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)


In [2]:

#data handling
df = pd.read_csv('EMAIL.csv')

print("Dataset:")
print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nClass Distribution: \n")
print(df["label"].value_counts())



Dataset:
                                  email label
0  Congratulations you won a free prize  spam
1            Claim your cash reward now  spam
2       Win money by clicking this link  spam
3                You have won a lottery  spam
4             Free gift waiting for you  spam

Dataset Shape:
(20, 2)

Missing Values:
email    0
label    0
dtype: int64

Class Distribution: 

spam    10
ham     10
Name: label, dtype: int64


In [3]:
#droping duplicate values
df = df.drop_duplicates()

# Remove missing values
df = df.dropna()

# Convert text to lowercase
df["email"] = df["email"].str.lower()

# Separate input and output
X = df["email"]
y = df["label"]


In [5]:
#data spliting
X_train, X_test, y_train, y_test = train_test_split( X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)
#data size 
print("\nTraining data:", len(X_train))
print("Testing data:", len(X_test))



Training data: 15
Testing data: 5


In [6]:
#Convert text into numerical features
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Training Shape:")
print(X_train_tfidf.shape)




TF-IDF Training Shape:
(15, 55)


In [8]:
#model selection and training
model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("\nModel Training Completed!")



Model Training Completed!


In [9]:
y_pred=model.predict(X_test_tfidf)

In [10]:


accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test, y_pred, pos_label="spam",
    zero_division=0
)

recall = recall_score(
    y_test, y_pred, pos_label="spam",
    zero_division=0
)

f1 = f1_score(
    y_test, y_pred, pos_label="spam",
    zero_division=0
)

#print("\n========== MODEL EVALUATION ==========")

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test, y_pred,
    labels=["ham", "spam"]
))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    zero_division=0
))


Accuracy: 80.0 %
Precision: 0.67
Recall: 1.0
F1 Score: 0.8

Confusion Matrix:
[[2 1]
 [0 2]]

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      0.67      0.80         3
        spam       0.67      1.00      0.80         2

    accuracy                           0.80         5
   macro avg       0.83      0.83      0.80         5
weighted avg       0.87      0.80      0.80         5



In [20]:
print("\n========== EMAIL SPAM CHECKER ==========")

spam_sender = [
    'fake@gmail.com',
    'scammer@gmail.com'
]

spam_email = []
not_spam_email = []

# Common spam words
spam_words = [
    'win',
    'winner',
    'free',
    'prize',
    'offer',
    'urgent',
    'money',
    'click here',
    'lottery',
    'congratulations'
]

# Get spam emails from the dataset
spam_dataset = df[
    df['label'].str.lower() == 'spam'
]['email'].str.lower().tolist()

# Combine dataset spam emails and spam words
spam_check = spam_dataset + spam_words


while True:

    sender = input(
        "\nEnter the sender address or type 'exit': "
    ).lower().strip()

    # Exit program
    if sender == 'exit':
        print("\nExiting Spam Checker...")
        print("Thank you for using the Spam Checker!")
        break

    # Check sender address
    if sender in spam_sender:

        print("\nEmail Status = SPAM")
        print("Sender:", sender)

        spam_email.append({
            'sender': sender,
            'email': 'Sender found in spam list',
            'status': 'SPAM'
        })

    else:

        # Get email message
        email = input(
            "\nEnter email message or type 'exit': "
        ).lower().strip()

        # Exit program
        if email == 'exit':
            print("\nExiting Spam Checker...")
            print("Thank you for using the Spam Checker!")
            break

        # Check email against spam dataset and spam words
        if any(word in email for word in spam_check):

            print("\nEmail Status = SPAM")

            spam_email.append({
                'sender': sender,
                'email': email,
                'status': 'SPAM'
            })

        else:

            print("\nEmail Status = NOT SPAM")

            not_spam_email.append({
                'sender': sender,
                'email': email,
                'status': 'NOT SPAM'
            })


# ==================================================
# CREATE DATAFRAMES
# ==================================================

spam_df = pd.DataFrame(spam_email)
not_spam_df = pd.DataFrame(not_spam_email)

# Display Spam emails
print("\n========== SPAM EMAILS ==========")
print(spam_df)

# Display Not Spam emails
print("\n========== NOT SPAM EMAILS ==========")
print(not_spam_df)


# ==================================================
# COMBINE BOTH DATAFRAMES
# ==================================================

all_email_df = pd.concat(
    [spam_df, not_spam_df],
    ignore_index=True
)

print("\n========== ALL EMAIL DETAILS ==========")
print(all_email_df)


# Save results to CSV
all_email_df.to_csv(
    "email_spam_results.csv",
    index=False
)

print("\nResults saved to email_spam_results.csv")


========== EMAIL SPAM CHECKER ==========



Enter the sender address or type 'exit':  fake@gmail.com



Email Status = SPAM
Sender: fake@gmail.com



Enter the sender address or type 'exit':  prajwal45@gmil.com

Enter email message or type 'exit':  hlo



Email Status = NOT SPAM



Enter the sender address or type 'exit':  exit



Exiting Spam Checker...
Thank you for using the Spam Checker!

========== SPAM EMAILS ==========
           sender                      email status
0  fake@gmail.com  Sender found in spam list   SPAM

========== NOT SPAM EMAILS ==========
               sender email    status
0  prajwal45@gmil.com   hlo  NOT SPAM

========== ALL EMAIL DETAILS ==========
               sender                      email    status
0      fake@gmail.com  Sender found in spam list      SPAM
1  prajwal45@gmil.com                        hlo  NOT SPAM

Results saved to email_spam_results.csv
